# Lesson 5 — The 3 AM Page

> **Case study.** PagerDuty fires: *"Refund agent error rate spiked 40x in 
> the last 10 minutes."* You are on-call. You have one log to triage: 
> `logs/policy-audit.jsonl`. You need to know *who*, *what*, *why* — fast.

## Learning objectives
1. Generate a realistic stream of mixed allow/deny decisions.
2. Load and aggregate the audit log with pandas.
3. Reproduce a denial deterministically with `opa.evaluate(...)`.
4. Sketch a Rego fix and re-run.


In [1]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
os.chdir(ROOT)               # so ./logs/policy-audit.jsonl lands in the standard location
sys.path.insert(0, str(ROOT))
from src.opa import OPAClient
opa = OPAClient()
assert opa.health_check(), 'OPA server unreachable on http://localhost:8181. Run: docker-compose up -d opa'
print('OPA reachable ✓   cwd =', pathlib.Path.cwd())

OPA reachable ✓   cwd = /home/anirudh/Projects/Rogers/agent-policy-opa


## 1. Generate a realistic incident stream

In [2]:
# Start with a clean audit log so we only see the incident we generate.
(ROOT / 'logs' / 'policy-audit.jsonl').write_text('')

from src.agents import ReturnsAgent
from src.models import AgentRole

cashier = ReturnsAgent(role=AgentRole.CASHIER)
mgr     = ReturnsAgent(role=AgentRole.STORE_MANAGER)

# 20 normal refunds (allowed)
for _ in range(20):
    cashier.invoke_tool('process_refund', {'order_id': 'ORD-001', 'amount': 50.0})
# 10 over-cap attempts (OPA-A denial)
for _ in range(10):
    cashier.invoke_tool('process_refund', {'order_id': 'ORD-001', 'amount': 250.0})
# 5 no-receipt large refunds (OPA-A then OPA-B)
for _ in range(5):
    mgr.invoke_tool('process_refund', {'order_id': 'ORD-002', 'amount': 89.99})
# 3 huge refunds
for _ in range(3):
    cashier.invoke_tool('process_refund', {'order_id': 'ORD-003', 'amount': 1299.50})
print('Incident stream written to audit log.')

Incident stream written to audit log.


## 2. Load + aggregate

In [3]:
import json, pandas as pd
audit_path = ROOT / 'logs' / 'policy-audit.jsonl'
with audit_path.open() as f:
    entries = [json.loads(l) for l in f if l.strip()]
df = pd.json_normalize(entries)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['allow'] = df['allow'].astype(bool)
df = df.sort_values('timestamp').tail(60).reset_index(drop=True)  # focus on the recent burst
print(df[['timestamp', 'agent_id', 'policy_path', 'allow', 'reason']].tail(10).to_string(index=False))

                 timestamp    agent_id          policy_path  allow reason
2026-05-28 21:49:40.141904 returns-001 retail/authorization  False   None
2026-05-28 21:49:40.144787 returns-001 retail/authorization  False   None
2026-05-28 21:49:40.148805 returns-002 retail/authorization  False   None
2026-05-28 21:49:40.151362 returns-002 retail/authorization  False   None
2026-05-28 21:49:40.153768 returns-002 retail/authorization  False   None
2026-05-28 21:49:40.156040 returns-002 retail/authorization  False   None
2026-05-28 21:49:40.158239 returns-002 retail/authorization  False   None
2026-05-28 21:49:40.160573 returns-001 retail/authorization  False   None
2026-05-28 21:49:40.162948 returns-001 retail/authorization  False   None
2026-05-28 21:49:40.167224 returns-001 retail/authorization  False   None


## 3. Top deny reasons

In [4]:
denials = df[df['allow'] == False].copy()
summary = denials.groupby(['policy_path', 'reason']).size().sort_values(ascending=False)
print(summary.to_string())

Series([], )


## 4. Reproduce one denial deterministically

In [5]:
# json_normalize flattened the nested input_data; rebuild it from the raw entries.
raw_denials = [e for e in entries if not e['allow']]
sample = raw_denials[-1]
print('Replaying denial for agent', sample['agent_id'])
decision = opa.evaluate(sample['policy_path'], sample['input_data'])
print('allow :', decision.allow)
print('reason:', decision.reason)

Replaying denial for agent returns-001
allow : False
reason: None


## 5. Hypothetical Rego fix

Suppose product told you to lift the cashier refund cap for gold-tier customers. 
You would add (do **not** ship without code review):

```rego
allow if {
    input.action == "process_refund"
    input.agent.role == "cashier"
    input.resource.amount <= 300
    input.resource.customer_tier == "gold"
    input.resource.days_since_purchase <= 30
}
```
and pass `customer_tier` in the tool's `enforce(...)` resource dict.


## 6. Per-minute deny rate

In [6]:
rate = (denials.set_index('timestamp')
                 .resample('1min')
                 .size()
                 .rename('denies_per_min'))
print(rate.tail(10).to_string())

timestamp
2026-05-28 21:49:00    38
Freq: min


## Recap
* Every decision is a single JSON line — `jq` / `pandas` friendly.
* Reproducing a denial is one `opa.evaluate(...)` away.
* Fixes ship as **Rego edits**, not Python releases.

**Next:** [06 — Shipping to Production](./06-foundry-deployment.ipynb).
